# クイズ / ブログ用グラフ生成

旧05 + 旧06/10のグラフ機能を統合。用途に応じてセルを選んで実行する。

- **出題用**: タイトルを伏せたダークテーマ2段グラフ（9:16 / 4:5）
- **回答・ブログ用**: 累積 / 日次 / 期間ズームの単発グラフ（テーマ選択可）

In [ ]:
#@title 🔧 セットアップ（パッケージinstall + Driveマウント）
GITHUB_OWNER = "asmrt"  # ← GitHubユーザー名に変更

from google.colab import userdata, drive
token = userdata.get("GITHUB_TOKEN")
!pip install -q "git+https://{token}@github.com/{GITHUB_OWNER}/unofficial_sixfonia_analytics.git"
!pip install -q japanize-matplotlib
drive.mount("/content/drive")

from sixfonia_analytics import plots
plots.setup_japanese_font()

In [ ]:
#@title ⚙️ 設定と対象動画データの構築
CHANNEL = "hima72"  #@param ["hima72", "sixfonia", "kosame", "illuma", "mikoto", "suchi", "lan"]
VIDEO_ID = "PkxauVaqgI0"  #@param {type:"string"}

from sixfonia_analytics import load, metrics

combined_df = load.build_combined_df(CHANNEL)
video_df = metrics.filter_video(combined_df, VIDEO_ID)
print(f"対象データ: {len(video_df)} 日分")
video_df.tail(3)

In [ ]:
#@title 🎯 出題用グラフ（タイトル伏せ / ダークテーマ）
QUIZ_VOL = "vol.1"  #@param {type:"string"}
LAYOUT = "4x5"  #@param ["4x5", "9x16"]
ANSWER_NOTE = "正解は後日発表！"  #@param {type:"string"}
#@markdown 伏せタイトル（実タイトルの文字数に合わせて ? を調整）
MASKED_TITLE = "「【???????】?????????【????????】」"  #@param {type:"string"}
#@markdown 基準線（例: 100000。0で非表示）
REF_LINE = 100000  #@param {type:"integer"}

plots.plot_quiz_graph(
    video_df,
    layout=LAYOUT,
    header_title=f"シクフォニ再生数クイズ（非公式）{QUIZ_VOL}",
    masked_title=MASKED_TITLE,
    footer_text=ANSWER_NOTE,
    ref_line=REF_LINE or None,
    save_path=f"quiz_graph_{LAYOUT}.png",
)

In [ ]:
#@title 📝 回答・ブログ用グラフ（累積 / 日次 / 期間ズーム）
THEME = "blog_dark"  #@param ["blog_dark", "blog_light", "quiz"]
#@markdown 期間ズーム（空で全期間。例: 2026-03-23）
DATE_FROM = ""  #@param {type:"string"}
DATE_TO = ""  #@param {type:"string"}

date_range = (DATE_FROM, DATE_TO) if DATE_FROM and DATE_TO else None

plots.plot_single_trend(video_df, kind="cumulative", theme=THEME,
                        date_range=date_range, save_path="blog_cumulative.png")
plots.plot_single_trend(video_df, kind="daily", theme=THEME,
                        date_range=date_range, save_path="blog_daily.png")

In [ ]:
#@title 📉 欠測日を補間した日次グラフ（データに抜けがある期間用・旧05）
USE_INTERPOLATION = False  #@param {type:"boolean"}

if USE_INTERPOLATION:
    series = metrics.prepare_continuous_series(video_df, value_col="viewCountDiff",
                                               date_range=date_range)
    plots.plot_interpolated_series(series, "viewCountDiff",
                                   title=f"Daily views (interpolated) — {VIDEO_ID}")
else:
    print("USE_INTERPOLATION が False のためスキップ")

In [ ]:
#@title ⬇️ 生成した画像のダウンロード
import glob
from google.colab import files

for f in glob.glob("quiz_graph_*.png") + glob.glob("blog_*.png"):
    files.download(f)